# Beyond linear regression: giving the model layers

In `pytorch_basics.ipynb` our "model" was really just a parameterized line: `y = weight * x + bias`. No matter how long we trained it, it could only ever learn straight lines.

Here we stack linear layers with a **nonlinearity (ReLU)** between them and see what changes:

1. the same synthetic line data still fits (sanity check)
2. a genuinely **nonlinear** pattern (a curve) becomes learnable — this is the whole point of "deep" learning

Companion to `pytorch_basics.ipynb`: same `device` setup, same 80:20 split, same `plot_predictions()` helper.

In [1]:
import torch
import matplotlib.pyplot as plt
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. The data

A. the **line** from the previous notebook (`y = 0.7x + 0.3`) — so any change comes from the model, not the data
B. a **curve**: `y = 0.7 * x^2 + 0.3`, plus a small wiggle for realism

In [3]:
# --- A) the line we already know how to fit ---
weight = 0.7
bias = 0.3

start = 0
end = 1
step = 0.02
X = torch.arange(start, end, step).unsqueeze(dim=1)
y = weight * X + bias

X[:5], X.shape

In [4]:
# --- B) a curve the old model could never fit ---
X_curve = torch.arange(0, 1, 0.02).unsqueeze(dim=1)
y_curve = weight * X_curve**2 + bias

X_curve[:5], X_curve.shape

In [5]:
# split the data into training and test set by 80:20 rule
train_split = int(0.8 * len(X))
X_train, y_train = X[:train_split].to(device), y[:train_split].to(device)
X_test, y_test = X[train_split:].to(device), y[train_split:].to(device)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [6]:
def plot_predictions(train_data=X_train.cpu(),
                     train_labels=y_train.cpu(),
                     test_data=X_test.cpu(),
                     test_labels=y_test.cpu(),
                     predictions=None):
    """Plots training data, test data and model predictions (on cpu, for matplotlib)."""
    plt.figure(figsize=(10, 7))
    plt.scatter(train_data, train_labels, c="b", s=8, label="Training data")
    plt.scatter(test_data, test_labels, c="g", s=8, label="Testing data")
    if predictions is not None:
        plt.scatter(test_data, predictions, c="r", s=8, label="Predictions")
    plt.legend(prop={"size": 14})
    plt.show()

plot_predictions()

## 2. The model: layers + ReLU instead of one weight and one bias

Compare with the old model:

```python
# before: 2 parameters total — can only output a straight line
self.weight = nn.Parameter(torch.randn(1, requires_grad=True))
self.bias = nn.Parameter(torch.randn(1, requires_grad=True))
return self.weight * x + self.bias
```

```python
# now: 3 linear layers with ReLU between them
self.layer1 = nn.Linear(in_features=1, out_features=16)
self.layer2 = nn.Linear(in_features=16, out_features=16)
self.layer3 = nn.Linear(in_features=16, out_features=1)
self.relu = nn.ReLU()
```

**ReLU is the piece that was missing.** It is not linear — it zeroes out negative values, so the composition of layers stops being another straight line. Without it, three stacked `nn.Linear` layers collapse to one linear function no matter how many there are.

Each `nn.Linear(in, out)` holds `in*out` weights + `out` biases, randomly initialized, all learnable via backpropagation. Total parameters this time: `1*16 + 16` + `16*16 + 16` + `16*1 + 1` = **337 + 24 = 361**, up from 2.

In [8]:
torch.manual_seed(37)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=1, out_features=16)
        self.layer2 = nn.Linear(in_features=16, out_features=16)
        self.layer3 = nn.Linear(in_features=16, out_features=1)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layer3(self.relu(self.layer2(self.relu(self.layer1(x)))))

model = MLP().to(device)
model

In [9]:
# count parameters — the old model had 2
sum(p.numel() for p in model.parameters())

In [10]:
# what does an untrained layered model predict? not a straight line anymore.
with torch.inference_mode():
    untrained_preds = model(X_test).cpu()
plot_predictions(predictions=untrained_preds)

## 3. Training on the line — same loop, only the optimizer differs

Same 5 steps as always:
1. forward pass
2. compute loss (`nn.L1Loss`)
3. `optimizer.zero_grad()`
4. `loss.backward()`
5. `optimizer.step()`

We also track loss per epoch this time, so we can plot the curve afterwards.

In [12]:
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)
epochs = 500

loss_history = []

for epoch in range(epochs):
    model.train()

    y_pred = model(X_train)                # 1. forward pass
    loss = loss_fn(y_pred, y_train)        # 2. compute loss
    optimizer.zero_grad()                  # 3. prep for backward pass
    loss.backward()                        # 4. backpropagation
    optimizer.step()                       # 5. gradient descent

    loss_history.append(loss.item())

print(f"epoch: {epoch} loss: {loss.item():.4f}")

epoch: 499 loss: 0.0152


In [13]:
plt.figure(figsize=(10, 4))
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("L1 loss")
plt.title("Training loss on the line data")
plt.show()

### Inference

Remember to switch to `model.eval()` and use `torch.inference_mode()` so gradients aren't tracked.

In [15]:
model.eval()
with torch.inference_mode():
    y_pred = model(X_test).cpu()

plot_predictions(predictions=y_pred)

### Sanity check: it fits the line just as well

If it hasn't, that's a signal something else got worse (probably learning rate or epochs) — not that layers are wrong.

## 4. Training on the curve — where layers actually matter

Swap in `X_curve` / `y_curve`, split the same way, and train. Expect the old linear-only model to *fail* here — intuition: \(y = 0.7x^2 + 0.3\) is not a straight line, so 2 parameters can't match it.

This is exactly why the notebook's title is "beyond linear regression": the previous notebook could represent only lines; now we're fitting a curve.

In [18]:
# split the curve data
train_split = int(0.8 * len(X_curve))
Xc_train, yc_train = X_curve[:train_split].to(device), y_curve[:train_split].to(device)
Xc_test, yc_test = X_curve[train_split:].to(device), y_curve[train_split:].to(device)

plot_predictions(train_data=Xc_train.cpu(), train_labels=yc_train.cpu(),
                 test_data=Xc_test.cpu(), test_labels=yc_test.cpu())

In [19]:
# train the same MLP on the curve data
torch.manual_seed(37)
model_curve = MLP().to(device)
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(params=model_curve.parameters(), lr=0.005)
epochs = 2000

loss_history = []
for epoch in range(epochs):
    model_curve.train()
    y_pred = model_curve(Xc_train)
    loss = loss_fn(y_pred, yc_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

print(f"epoch: {epoch} loss: {loss.item():.4f}")

epoch: 1999 loss: 0.0052


In [20]:
plt.figure(figsize=(10, 4))
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("L1 loss")
plt.title("Training loss on the curve data")
plt.show()

In [21]:
model_curve.eval()
with torch.inference_mode():
    yc_pred = model_curve(Xc_test).cpu()

plot_predictions(train_data=Xc_train.cpu(), train_labels=yc_train.cpu(),
                 test_data=Xc_test.cpu(), test_labels=yc_test.cpu(),
                 predictions=yc_pred)

## 5. Troubleshooting: SGD vs Adam

Above, plain **SGD** with a small learning rate needed thousands of epochs and still plateaus at a visible loss. Switch the optimizer to **Adam**, bump the learning rate, and the same model fits the curve in far fewer epochs.

This isn't an "Adam is always better" rule — it's a preview of chapter 04's real lesson: **optimizer + learning rate are hyperparameters you tune, not constants**.

In [23]:
# same model, different optimizer: Adam instead of SGD
torch.manual_seed(37)
model_curve_adam = MLP().to(device)
loss_fn = nn.L1Loss()
optimizer = torch.optim.Adam(params=model_curve_adam.parameters(), lr=0.01)
epochs = 1000

loss_history_adam = []
for epoch in range(epochs):
    model_curve_adam.train()
    y_pred = model_curve_adam(Xc_train)
    loss = loss_fn(y_pred, yc_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history_adam.append(loss.item())

print(f"epoch: {epoch} loss: {loss.item():.4f}")

epoch: 999 loss: 0.0016


In [24]:
plt.figure(figsize=(12, 4))
plt.plot(loss_history, alpha=0.5)
plt.legend(["SGD"])
plt.show()

In [25]:
model_curve_adam.eval()
with torch.inference_mode():
    yc_pred_adam = model_curve_adam(Xc_test).cpu()
plot_predictions(train_data=Xc_train.cpu(), train_labels=yc_train.cpu(),
                 test_data=Xc_test.cpu(), test_labels=yc_test.cpu(),
                 predictions=yc_pred_adam)

## 6. What we learned

1. `nn.Linear` layers are interchangeable building blocks; `nn.ReLU` between them is what gives the model the ability to curve (without it, stacked layers still collapse to a line)
2. the training loop is unchanged — same 5 steps, same loss function; only the model and data changed
3. optimizer + learning rate are **hyperparameters**: SGD needed many more epochs than Adam for the same fit
4. inference needs `.eval()` + `torch.inference_mode()`

Next: **classification** — instead of predicting a number, predict a category, which needs `nn.BCEWithLogitsLoss`, accuracy as a metric, and circle toy data to make the problem actually hard.